In [2]:
AttachSpec("diffalg.spec");
Z:=Integers();
Q:=RationalField();

R<t>:=PolynomialRing(Q,1);
f := map<R->R|f:->0>;
A<t> := DifferentialRing(R, f, Q);
F:=FieldOfFractions(A);
P<x>:=PolynomialRingProlSeq(F,1: term_order:=<"dblocks",[[1]]>);

In [3]:
function trace_ideal(eqns,ord:tolerance:=5)
    assert #eqns gt 0;
    P:=Parent(eqns[1]);
    J_low:=Jet(P,ord);
    r:=ord+tolerance;
    diff_eqns:=[Diff(f,j) : f in eqns, j in [0..r]];
    r0:=Order(diff_eqns);
    Ir0:=ideal<Jet(P,r0)|Jet(diff_eqns)>;
    //higher_derivatives:=[Diff(P.i,j) : j in [ord+1..r0], i in [1..Ngens(P)]];
    //higher_derivatives_r0:=Jet(higher_derivatives,r0);
    lower_derivatives:=[Diff(P.i,j) : j in [0..ord], i in [1..Ngens(P)]];
    lower_derivatives_r0:=Jet(lower_derivatives,r0);
    
    EI0:=EliminationIdeal(Ir0,{v: v in lower_derivatives_r0});
    lower_derivatives:=Jet(lower_derivatives,ord);
    zeros:=[J_low!0 : j in [ord+1..r0], i in [1..Ngens(P)]];
    I_low:=ideal<J_low|[Evaluate(g,lower_derivatives cat zeros) : g in Generators(EI0)]>;
    return I_low;
end function;

function trace_monomial_ideal_step(eqns0,ord:tolerance:=5)
    assert #eqns0 gt 0;
    P:=Parent(eqns0[1]);
    J_low:=Jet(P,ord);
    
    I_ord:=trace_ideal(eqns0,ord:tolerance:=tolerance);
    MI_ord:=LeadingMonomialIdeal(I_ord);
    eqns:=[P!g : g in Generators(MI_ord)];
    
    
    r:=ord+tolerance;
    diff_eqns:=[Diff(f,j) : f in eqns, j in [0..r]];
    r0:=Order(diff_eqns);
    eqns_r0 :=&cat[Monomials(g) : g in Jet(diff_eqns)];
    Ir0:=ideal<Jet(P,r0)|eqns_r0>;
    
    lower_derivatives:=[Diff(P.i,j) : j in [0..ord], i in [1..Ngens(P)]];
    lower_derivatives_r0:=Jet(lower_derivatives,r0);
    
    EI0:=EliminationIdeal(Ir0,{v: v in lower_derivatives_r0});
    lower_derivatives:=Jet(lower_derivatives,ord);
    zeros:=[J_low!0 : j in [ord+1..r0], i in [1..Ngens(P)]];
    I_low:=ideal<J_low|[Evaluate(g,lower_derivatives cat zeros) : g in Generators(EI0)]>;
    return I_low;
end function;

function trace_monomial_ideal(eqns,ord : max:=20, tolerance:=5)
    assert #eqns gt 0;
    P:=Parent(eqns[1]);
    MI0:=trace_monomial_ideal_step(eqns,ord:tolerance:=tolerance);
    eqns0:=[P!g : g in Generators(MI0)];
    MI1:=trace_monomial_ideal_step(eqns0,ord:tolerance:=tolerance);
    i:=1;
    while MI0 ne MI1 and i lt 10 do 
        i:=i+1;
        MI0:=MI1;
        eqns0:=[P!g : g in Generators(MI0)];
        MI1:=trace_monomial_ideal_step(eqns0,ord:tolerance:=tolerance);
    end while; 
    
    return MI0,i;
    
end function;

In [4]:
f:=Diff(x,1)-x^2-x-1;
eqns:=[f];
I5:=trace_ideal(eqns,5);
M5:=trace_monomial_ideal_step(eqns,5);
Dimension(M5);
Dimension(I5);

1 [ 1 ]
1 [ 6 ]


In [5]:
//this takes forever to run
f:=x^3;
eqns:=[f];
time I:=trace_ideal(eqns,4);
time M:=trace_monomial_ideal_step(eqns,4);
eqns1:=[P!g: g in Generators(M)];
M1:=trace_monomial_ideal_step(eqns1,4);

Time: 0.090
Time: 31.260


In [28]:
time M:=trace_monomial_ideal(eqns,4);
time M1:=trace_monomial_ideal(eqns,4:tolerance:=10);

Time: 27.020
Time: 156.450


In [17]:
Dimension(M);
Dimension(I); //the tolerance isn't big enough, this should be dimension zero.

0 []
1 [ 5 ]


In [23]:
time Dimension(trace_ideal(eqns,4:tolerance:=11));

0 []
Time: 6.700


In [52]:
eqns_jet:=Jet(&cat[Monomials(g) : g in eqns0]);

[
Diff(x,2)*Diff(x,3)^9,
Diff(x,2)*Diff(x,3)^8*Diff(x,4),
Diff(x,3)^10,
Diff(x,2)*Diff(x,3)^8*Diff(x,5),
Diff(x,2)*Diff(x,3)^7*Diff(x,4)^2,
Diff(x,3)^9*Diff(x,4),
Diff(x,2)*Diff(x,3)^8*Diff(x,6),
Diff(x,2)*Diff(x,3)^7*Diff(x,4)*Diff(x,5),
Diff(x,3)^9*Diff(x,5),
Diff(x,2)*Diff(x,3)^6*Diff(x,4)^3,
Diff(x,3)^8*Diff(x,4)^2,
Diff(x,2)*Diff(x,3)^8*Diff(x,7),
Diff(x,2)*Diff(x,3)^7*Diff(x,4)*Diff(x,6),
Diff(x,3)^9*Diff(x,6),
Diff(x,2)*Diff(x,3)^7*Diff(x,5)^2,
Diff(x,2)*Diff(x,3)^6*Diff(x,4)^2*Diff(x,5),
Diff(x,3)^8*Diff(x,4)*Diff(x,5),
Diff(x,2)*Diff(x,3)^5*Diff(x,4)^4,
Diff(x,3)^7*Diff(x,4)^3,
Diff(x,2)*Diff(x,3)^8*Diff(x,8),
Diff(x,2)*Diff(x,3)^7*Diff(x,4)*Diff(x,7),
Diff(x,3)^9*Diff(x,7),
Diff(x,2)*Diff(x,3)^7*Diff(x,5)*Diff(x,6),
Diff(x,2)*Diff(x,3)^6*Diff(x,4)^2*Diff(x,6),
Diff(x,3)^8*Diff(x,4)*Diff(x,6),
Diff(x,2)*Diff(x,3)^6*Diff(x,4)*Diff(x,5)^2,
Diff(x,3)^8*Diff(x,5)^2,
Diff(x,2)*Diff(x,3)^5*Diff(x,4)^3*Diff(x,5),
Diff(x,3)^7*Diff(x,4)^2*Diff(x,5),
Diff(x,2)*Diff(x,3)^4*Diff(x,4)^5,
D

In [28]:
MI:=LeadingMonomialIdeal(I5);
eqns_new:=[P!g : g in Generators(MI)];

In [29]:
eqns_new;

[
Diff(x,5),
Diff(x,4),
Diff(x,3),
Diff(x,2),
Diff(x,1)
]


In [32]:
I_new:=trace_ideal(eqns_new,3);
//MI_new:=LeadingMonomialIdeal(I_new);

In [ ]:
f:=Diff(x,2)*Diff(x,1)-x^3;
eqns:=[f];
time I5:=trace_ideal(eqns,5:tolerance:=6);
time M5:=trace_monomial_ideal(eqns,5:tolerance:=6);
Dimension(M5);
Dimension(I5);

Loading "/var/folders/9f/rk6hvc213hn7t38vt1p_wsyw0000gn/T/tmpp8njubjc"
